In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
import sentencepiece as spm
from tqdm.auto import tqdm

from configs.config import config

c:\Users\FH\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sp = spm.SentencePieceProcessor()
sp.load(str(config.tokenizer_path))

print("Vocabulary Size:", sp.vocab_size())


Vocabulary Size: 32000


In [4]:
input_file = config.train_file

output_file = (
    input_file.parent / "tokens.bin"
)

print(input_file)
print(output_file)

C:\Users\FH\Documents\PersionLLM\PersianGPT\data\processed\clean_corpus.txt
C:\Users\FH\Documents\PersionLLM\PersianGPT\data\processed\tokens.bin


In [5]:
sample = "سلام، امروز هوا خیلی خوب است."

tokens = sp.encode(sample)

print(tokens)
print(sp.decode(tokens))

[517, 31874, 612, 920, 611, 835, 37, 31867]
سلام، امروز هوا خیلی خوب است.


In [6]:
if output_file.exists():
    output_file.unlink()

print("Output:", output_file)

Output: C:\Users\FH\Documents\PersionLLM\PersianGPT\data\processed\tokens.bin


In [14]:
total_lines = 0
total_tokens = 0

with open(input_file, "r", encoding="utf-8") as fin, \
     open(output_file, "ab") as fout:

    for line in tqdm(fin):

        line = line.strip()

        if not line:
            continue

        token_ids = sp.encode(line)

        token_ids.append(sp.eos_id())

        np.asarray(
            token_ids,
            dtype=np.uint16
        ).tofile(fout)

        total_lines += 1
        total_tokens += len(token_ids)

print(f"Lines Processed : {total_lines:,}")
print(f"Total Tokens    : {total_tokens:,}")

4902568it [03:56, 20731.40it/s]

Lines Processed : 4,902,568
Total Tokens    : 134,955,913


In [19]:
print(f"File Size : {output_file.stat().st_size / 1024 / 1024:.2f} MB")

File Size : 505.47 MB


In [20]:
tokens = np.fromfile(
    output_file,
    dtype=np.uint16
)

print(tokens[:100])

[ 6083  1738    80 31879 15127 18341 11433  9144  6083  1738    80  3526
  2118  6307  6298  5850  4572  8947  3438  2776  6298  3249   523  2491
  6298  7369  1924    22  4915  2088  2548  4535 19839 31854 31886  4890
 13176 31952   469  5850  2303  8947  6307 16756  6298     9 31886  4890
 31866  1465  4572  8947   115    63   367 15368  2438 28827  1582   226
  2491    63    27  4535    58  3930   118 18914     9 17025  2579  3438
  2776    69    63 11501 11501  1536   226     9    58  3930  1224     9
  3526  2118  6307     9  5246    63 16721   226     9  1536   226   469
  5850  4535    63    27]


In [21]:
print("Number of Tokens :", len(tokens))

Number of Tokens : 265009258


In [22]:
print("BOS ID :", sp.bos_id())
print("EOS ID :", sp.eos_id())
print("UNK ID :", sp.unk_id())
print("PAD ID :", sp.pad_id())

BOS ID : 1
EOS ID : 2
UNK ID : 0
PAD ID : 3
